<a href="https://colab.research.google.com/github/dranphphmithe-ux/Book-Rental-System-Project/blob/Kwang/%E0%B8%AA%E0%B9%88%E0%B8%A7%E0%B8%99%E0%B8%82%E0%B8%AD%E0%B8%87%E0%B8%81%E0%B8%A7%E0%B8%B2%E0%B8%87%E0%B9%80%E0%B8%8A%E0%B9%87%E0%B8%81%E0%B9%80%E0%B8%A3%E0%B8%B5%E0%B8%A2%E0%B8%9A%E0%B8%A3%E0%B9%89%E0%B8%AD%E0%B8%A2%E0%B9%81%E0%B8%A5%E0%B9%89%E0%B8%A7%E0%B8%88%E0%B9%89%E0%B8%B2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ของกวางน้าเตง

In [ ]:
#ของกวาง
import csv
import io
import ssl
from urllib.request import Request, urlopen
# ==================================================
# 1. Class Book (โครงสร้างข้อมูลหนังสือ)
# ==================================================
class Book:
    """คลาสสำหรับเก็บข้อมูลหนังสือการ์ตูนแต่ละเล่ม"""

    #ส่วนรับข้อมูลเข้า(Input)เวลาสร้างหนังสือเล่มใหม่
    def __init__(
        self,
        book_isbn: str,          #รหัสประจำตัวหนังสือ (ฟีลเลขบัตรประจำตัวประชาชนเรา)
        title: str,              #ชื่อเรื่องของหนังสือ
        author: str,             #ชื่อผู้แต่ง
        price: str | float,      #ราคาปกหนังสือ
        category_id: str,        #รหัสประจำหนังสือของหมวดหมู่หลัก(เช่น มังงะ=01, การ์ตูนความรู้=02
        sub_category_id: str,    #รหัสประจำหนังสือของหมวดหมู่ย่อย(เช่น แนวแอ็กชัน = 01, แนวสืบสวน  = 02 )
        shelf_location: str,     #ตำแหน่งของหนังสือที่ชั้นวางหนังสือ(เช่น A-01 ,B-01)
        stock_qty: str | int,    #จำนวนหนังสือที่มีอยู่ในstock(stockในร้านมี=5เล่ม)
    ):
        # self = แปะข้อมูลลงในหนังสือเล่มนี้
        # .strip() = ลบช่องวางส่วนเกินออก เพื่อให้ค้นหาง่าย
        self.book_isbn = str(book_isbn).strip()
        self.title = str(title).strip()
        self.author = str(author).strip()

        # แปลงราคาเป็นตัวเลข(float)ไว้คำนวณเงินและกันโปรแกรมเจ๊งถ้าราคาผิด
        try:
            self.price = float(price) if price else 0.0
        except ValueError:
            self.price = 0.0

        # แปลงราคาจำนวณสต็อกเป็นตัวเลขจำนวยเต็ม(int)ไว้ใช้ตัดสต็อกเวลา ยืม-คืน
        try:
            self.stock_qty = int(stock_qty) if stock_qty else 0
        except ValueError:
            self.stock_qty = 0

        self.category_id = str(category_id).strip()
        self.sub_category_id = str(sub_category_id).strip()
        self.shelf_location = str(shelf_location).strip()

    # ส่วนเช็กหนังสือ
    @property
    def status(self) -> str:
        #  เช็กว่าสต็อกเหลือมั้ย
        """สถานะการเช่า: ถ้าสต็อก > 0 เป็น 'พร้อมเช่า' ถ้าสต็อก <= 0 เป็น 'สินค้าหมด'"""
        return "พร้อมเช่า" if self.stock_qty > 0 else "สินค้าหมด"

    def is_available(self) -> bool:
        # เช็กสั้นๆ True/False ว่าเล่มนี้พร้อมให้ยืมมั้ย
        return self.stock_qty > 0

    # ส่วนจัดการตัด/เพิ่มสต็อกหนังสือ
    def decrease_stock(self, qty: int = 1) -> bool:
        # ฟังก์ชันตัดสต็อกเวลาคนมายืม(ค่าเริ่มต้นลบทีละ1)
        if self.stock_qty >= qty:
            self.stock_qty -= qty     # ถ้าสต็อกพอจะหักออก
            return True       #ตัดสำเร็จ
        return False          #ถ้าสต็อกไม่พอจะ ตัดไม่สำเร็จ

    def increase_stock(self, qty: int = 1):
        # ฟัวก์ชันเพิ่มสต็อกเวลาคนเอาหนังสือมาคืน (+ทีละ1)
        self.stock_qty += qty